This script appends the h2-distribution matrix to the IAM output file

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# Load matrix and IAM output file
df_h2 = pd.read_excel(
    r"C:\Users\ac145674\Downloads\h2_distribution_matrix_IAM_by-year.xlsx",
    sheet_name="h2-distribution",
 )
#
# df_iam = pd.read_csv(
#    r"C:\Users\ac145674\Downloads\message_SSP2-M.csv"
# )



In [4]:
# -- Import any file in the iam_output_files folder, to which the distribution matrix will be appended to. --
def load_iam_file(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == ".csv":
        return pd.read_csv(filepath)
    elif ext == ".mif":
        return pd.read_csv(filepath, sep=";")
    elif ext == ".xlsx":
        return pd.read_excel(filepath)
    else:
        raise ValueError(f"Unsupported file format: {ext}")


supported = [".csv", ".mif", ".xlsx"]
input_folder = Path(r"C:\Users\ac145674\coding_projects\premise_h2-distribution\premise\data\iam_output_files")

iam_file = next(
    (f for f in input_folder.iterdir() if f.suffix.lower() in supported), None
)

if iam_file is None:
    raise FileNotFoundError("No supported IAM file found in input folder")

df_iam = load_iam_file(iam_file)
print(f"Loaded: {iam_file.name}")



Loaded: REMIND_SSP2-NDC.mif


In [5]:
# ── Fill Scenario and Model from iam output file to h2 distribution matrix ──────────────────────────────────────────────
# Model and Scenario are constant across all iam rows
# quick sanity check, so there is only one scenario per iam-output-file.
assert (
    df_iam["Scenario"].nunique() == 1
), f"Expected 1 scenario, found {df_csv['Scenario'].nunique()}"
model_value = df_iam["Model"].dropna().iloc[0] 
scenario_value = (df_iam["Scenario"].dropna().iloc[0])  # could also use .unique()[0]

In [6]:
regions = sorted(df_iam["Region"].dropna().unique())
regions

['CAZ',
 'CHA',
 'EUR',
 'IND',
 'JPN',
 'LAM',
 'MEA',
 'NEU',
 'OAS',
 'REF',
 'SSA',
 'USA',
 'World']

In [7]:
# -- some data analysis --
regions = sorted(df_iam["Region"].dropna().unique())
scenario = sorted(df_iam["Scenario"].dropna().unique())
model = sorted(df_iam["Model"].dropna().unique())
variables = sorted(df_iam["Variable"].dropna().unique())
# regions
# scenario
# model

# -- looking for specific variables --
# df_iam[df_iam["Variable"] == 'FE|Transport|Bunkers|.*']
#df_iam[df_iam["Variable"].str.contains("FE.*Transport.*Bunkers")]
variables = sorted(
    df_iam[df_iam["Variable"].str.contains("Hydrogen")]["Variable"]
    .dropna()
    .unique()
)
variables

['Annualized fleet investments|Transport with bunkers|Hydrogen',
 'Annualized fleet investments|Transport|Hydrogen',
 'Annualized fleet investments|Transport|Pass with bunkers|Hydrogen',
 'Annualized fleet investments|Transport|Pass|Aviation|Hydrogen',
 'Annualized fleet investments|Transport|Pass|Domestic Aviation|Hydrogen',
 'Annualized fleet investments|Transport|Pass|Hydrogen',
 'Cap (GWel)|Hydrogen|Electricity',
 'Cap|Electricity|+|Hydrogen',
 'Cap|Gases|+|Hydrogen',
 'Cap|Hydrogen',
 'Cap|Hydrogen|+|Biomass',
 'Cap|Hydrogen|+|Coal',
 'Cap|Hydrogen|+|Electricity',
 'Cap|Hydrogen|+|Gas',
 'Cap|Hydrogen|Biomass|+|w/ CC',
 'Cap|Hydrogen|Biomass|+|w/o CC',
 'Cap|Hydrogen|Coal|+|w/ CC',
 'Cap|Hydrogen|Coal|+|w/o CC',
 'Cap|Hydrogen|Fossil',
 'Cap|Hydrogen|Fossil|+|w/ CC',
 'Cap|Hydrogen|Fossil|+|w/o CC',
 'Cap|Hydrogen|Gas|+|w/ CC',
 'Cap|Hydrogen|Gas|+|w/o CC',
 'Cap|Liquids|+|Hydrogen',
 'Carbon Management|Carbon Capture|Energy|Pe2Se|Biomass|+|Hydrogen w/ couple prod',
 'Carbon Manag

In [15]:
# some more data analysis
cols = ["Variable"] + [
    col
    for col in df_iam.columns
    if str(col) in [str(y) for y in range(2025, 2101)]
]
# df_iam[df_iam["Variable"].str.contains("Hydrogen")][cols]
df_iam[
    (df_iam["Variable"].str.contains("SE.*Hydrogen"))
    & (df_iam["Region"] == "EUR")
]

df_iam[
    (df_iam["Variable"] == "SE|Hydrogen")
    #& (df_iam["Region"] == "EUR")
]

,Model,Scenario,Region,Variable,Unit,2005,2010,2015,2020,2025,...,2050,2055,2060,2070,2080,2090,2100,2110,2130,2150
42793,REMIND,SSP2-NDC,CAZ,SE|Hydrogen,EJ/yr,0.003331,0.003921,0.004110,0.035249,0.048474,...,0.669790,0.727713,0.772282,0.708480,0.584353,0.569720,0.648966,0.708092,0.589795,0.346479
42794,REMIND,SSP2-NDC,CHA,SE|Hydrogen,EJ/yr,0.000078,0.000124,0.000536,0.053108,0.064909,...,1.903126,2.486926,3.343021,3.873991,4.159395,4.495236,5.220359,6.168307,5.984805,3.086435
42795,REMIND,SSP2-NDC,EUR,SE|Hydrogen,EJ/yr,0.025318,0.028293,0.029058,0.075785,0.090610,...,0.966539,1.112879,1.410040,1.567301,1.797885,2.086984,2.510353,2.698869,1.941566,0.782587
42796,REMIND,SSP2-NDC,IND,SE|Hydrogen,EJ/yr,0.000005,0.000012,0.000021,0.015982,0.023720,...,2.172018,2.845563,3.654708,4.225197,4.736020,5.447534,6.318635,7.612342,8.967602,6.008986
42797,REMIND,SSP2-NDC,JPN,SE|Hydrogen,EJ/yr,0.000794,0.000858,0.003777,0.007884,0.009409,...,0.180712,0.217177,0.276423,0.272792,0.236899,0.220894,0.240593,0.297343,0.273785,0.127241
42798,REMIND,SSP2-NDC,LAM,SE|Hydrogen,EJ/yr,0.000869,0.001163,0.001722,0.046336,0.047151,...,0.921190,1.233317,1.676688,2.095794,2.501848,3.061604,3.735213,4.436256,4.268878,2.402982
42799,REMIND,SSP2-NDC,MEA,SE|Hydrogen,EJ/yr,0.000217,0.000294,0.010040,0.065673,0.117440,...,4.522030,5.366595,6.125951,6.166731,5.719525,5.808075,6.903439,8.077354,7.321960,4.380225
42800,REMIND,SSP2-NDC,NEU,SE|Hydrogen,EJ/yr,0.000723,0.000833,0.001309,0.006869,0.008703,...,0.180637,0.245682,0.320387,0.328409,0.331707,0.367219,0.415901,0.450879,0.518855,0.354520
42801,REMIND,SSP2-NDC,OAS,SE|Hydrogen,EJ/yr,0.000266,0.000337,0.000555,0.028903,0.037713,...,2.402643,3.409470,4.785569,6.098150,7.050677,8.024182,9.367788,11.653843,13.535582,7.829083
42802,REMIND,SSP2-NDC,REF,SE|Hydrogen,EJ/yr,0.000106,0.003378,0.019042,0.035179,0.064828,...,0.999320,1.288939,1.712910,1.727069,1.667449,1.567975,1.502517,1.614973,1.591597,0.999608


In [39]:
# Make sure new variables from iam output matrix also have Model, Scenario and Region columns
df_h2 = df_h2.loc[df_h2.index.repeat(len(regions))].reset_index(drop=True)
df_h2["Region"] = regions * (len(df_h2) // len(regions))
df_h2["Model"] = model_value
df_h2["Scenario"] = scenario_value

In [40]:
# --- Drop index and 0 columns in the iam output file ---
# Rename integer year columns to strings so they align with the CSV
df_h2.columns = [str(c) for c in df_h2.columns]
# Drop the unnamed index column
if "Unnamed: 0" in df_iam.columns:
    df_iam = df_iam.drop(columns=["Unnamed: 0"])

In [41]:
# --- Cleaning columns ---
# Align column order
meta_cols = ["Model", "Scenario", "Region", "Variable", "Unit"]

# Union of year columns sorted numerically
year_cols_h2 = sorted([c for c in df_h2.columns if c.isdigit()], key=int)
year_cols_iam = sorted([c for c in df_iam.columns if c.isdigit()], key=int)
all_year_cols = sorted(set(year_cols_h2) | set(year_cols_iam), key=int)

# Reindex to full column set (NaN -> 0.0)
df_h2 = df_h2.reindex(columns=meta_cols + all_year_cols)
df_iam = df_iam.reindex(columns=meta_cols + all_year_cols)

In [42]:
# --- Merging dfs ---
df_merge = pd.concat([df_iam, df_h2], ignore_index=True)
# remove NaN in year columns
df_merge[all_year_cols] = df_merge[all_year_cols].fillna(0.0).astype(float)
df_merge

,Model,Scenario,Region,Variable,Unit,2005,2010,2015,2020,2025,...,2050,2055,2060,2070,2080,2090,2100,2110,2130,2150
0,REMIND,SSP2-NPi2025,CAZ,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,REMIND,SSP2-NPi2025,CHA,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,REMIND,SSP2-NPi2025,EUR,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,REMIND,SSP2-NPi2025,IND,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,REMIND,SSP2-NPi2025,JPN,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81901,REMIND,SSP2-NPi2025,OAS,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81902,REMIND,SSP2-NPi2025,REF,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81903,REMIND,SSP2-NPi2025,SSA,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81904,REMIND,SSP2-NPi2025,USA,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
# --- Export merged csv ---
# define output-path
output_path = iam_file
df_merge.to_csv(output_path, index=False)

# if input file was a .mif or .xlsx file:
ext = iam_file.suffix.lower()
if ext in [".csv", ".mif"]:
    sep = ";" if ext == ".mif" else ","
    df_merge.to_csv(iam_file, index=False, sep=sep)
elif ext == ".xlsx":
    df_merge.to_excel(iam_file, index=False)

# additional info
print(f"Table merged and save to {output_path}")
print(
    f"Shape: {df_merge.shape}  (expected rows added: {len(df_h2)} = {len(df_h2)//len(regions)} variables × {len(regions)} regions)"
)
print(f"\nModel:    {model_value}")
print(f"Scenario: {scenario_value}")
print(f"Regions:  {regions}")

Table merged and save to C:\Users\ac145674\coding_projects\premise_h2-distribution\premise\data\iam_output_files\REMIND_generic_SSP2-NPi2025.mif
Shape: (81906, 24)  (expected rows added: 585 = 45 variables × 13 regions)

Model:    REMIND
Scenario: SSP2-NPi2025
Regions:  ['CAZ', 'CHA', 'EUR', 'IND', 'JPN', 'LAM', 'MEA', 'NEU', 'OAS', 'REF', 'SSA', 'USA', 'World']
